In [3]:
import requests
import pandas as pd
from pathlib import Path

# API aadress
url = "https://previous-runs-api.open-meteo.com/v1/forecast"

# Ilmaennustusmudelid
models = {
    "ECMWF": "ecmwf_ifs025",
    "GFS": "ncep_gfs_global",
    "ICON_EU": "icon_eu"
}

# Ilmanäitajad (24 tundi ette tehtud prognoosid)
variables = [
    "temperature_2m_previous_day1",
    "precipitation_previous_day1",
    "wind_speed_10m_previous_day1"
]

kaust = Path.home() / "Downloads"
kaust.mkdir(parents=True, exist_ok=True)

# Siia kogume kõik andmed
all_data = []

# Aastad
for year in [2024, 2025, 2026]:

    # Ilmaennustusmudelid
    for name, model in models.items():

        params = {
            "latitude": 58.2642,
            "longitude": 26.4614,
            "start_date": f"{year}-06-01",
            "end_date": f"{year}-08-31",
            "hourly": ",".join(variables),
            "models": model,
            "timezone": "Europe/Tallinn",
            "wind_speed_unit": "ms"
        }

        print(f"Laadin: {name} {year}")

        # Küsime andmed Open-Meteost
        response = requests.get(
            url,
            params=params,
            timeout=120
        )
        response.raise_for_status()

        # JSON -> DataFrame
        data = response.json()
        df = pd.DataFrame(data["hourly"])

        # Lisame mudeli ja aasta
        df["model"] = name
        df["year"] = year

        # Salvestame eraldi faili
        filename = f"{name}_{year}.csv"
        df.to_csv(kaust / filename, index=False)

        # Lisame ühendatud andmestikku
        all_data.append(df)

        print(f"Salvestatud: {filename} ({len(df)} rida)")

# Ühendame kõik andmed
combined = pd.concat(all_data, ignore_index=True)

# Salvestame ühendatud faili
combined.to_csv(
    kaust / "all_forecasts.csv",
    index=False
)

print("\nVALMIS!")
print("Kokku ridu:", len(combined))
print("Failid asuvad:", kaust)

Laadin: ECMWF 2024
Salvestatud: ECMWF_2024.csv (2208 rida)
Laadin: GFS 2024
Salvestatud: GFS_2024.csv (2208 rida)
Laadin: ICON_EU 2024
Salvestatud: ICON_EU_2024.csv (2208 rida)
Laadin: ECMWF 2025
Salvestatud: ECMWF_2025.csv (2208 rida)
Laadin: GFS 2025
Salvestatud: GFS_2025.csv (2208 rida)
Laadin: ICON_EU 2025
Salvestatud: ICON_EU_2025.csv (2208 rida)
Laadin: ECMWF 2026
Salvestatud: ECMWF_2026.csv (2208 rida)
Laadin: GFS 2026
Salvestatud: GFS_2026.csv (2208 rida)
Laadin: ICON_EU 2026
Salvestatud: ICON_EU_2026.csv (2208 rida)

VALMIS!
Kokku ridu: 19872
Failid asuvad: C:\Users\nisu\Downloads


In [5]:
import pandas as pd
from pathlib import Path

kaust = Path.home() / "Downloads"

# 1. Funktsioon tegelike mõõtmiste lugemiseks
def loe_mootmised(fail, veeru_nimi):
    df = pd.read_csv(
        kaust / fail,
        sep=";",
        decimal=","
    )

    # Keskkonnaagentuuri kuupäevad on UTC ajas
    aeg_utc = pd.to_datetime({
        "year": df.iloc[:, 1],
        "month": df.iloc[:, 2],
        "day": df.iloc[:, 3],
        "hour": df.iloc[:, 4]
    }, utc=True)

    # Teisendame Eesti kohalikku aega
    aeg = aeg_utc.dt.tz_convert("Europe/Tallinn")

    tulemus = pd.DataFrame({
        "time": aeg.dt.tz_localize(None),
        veeru_nimi: df.iloc[:, 5]
    })

    # Ainult 2024–2026 suvekuud
    tulemus = tulemus[
        tulemus["time"].dt.year.isin([2024, 2025, 2026]) &
        tulemus["time"].dt.month.isin([6, 7, 8])
    ]

    return tulemus


# 2. Loeme tegelikud mõõtmised
temp = loe_mootmised("ohutemp.csv", "actual_temperature")
sademed = loe_mootmised("sademed.csv", "actual_precipitation")
tuul = loe_mootmised("tuulekiirus.csv", "actual_wind_speed")

# Ühendame kolm mõõtmist
actual = temp.merge(
    sademed, on="time", validate="one_to_one"
).merge(
    tuul, on="time", validate="one_to_one"
)

print("Tegelikud mõõtmised:", len(actual))


# 3. Loeme Open-Meteo prognoosid
forecasts = pd.read_csv(kaust / "all_forecasts.csv")
forecasts["time"] = pd.to_datetime(forecasts["time"])

# Muudame mudelid eraldi veergudeks
forecasts = forecasts.pivot(
    index="time",
    columns="model",
    values=[
        "temperature_2m_previous_day1",
        "precipitation_previous_day1",
        "wind_speed_10m_previous_day1"
    ]
)

# Lihtsustame veergude nimesid
nimed = {
    "temperature_2m_previous_day1": "temperature",
    "precipitation_previous_day1": "precipitation",
    "wind_speed_10m_previous_day1": "wind_speed"
}

forecasts.columns = [
    f"{nimed[muutuja]}_{mudel}"
    for muutuja, mudel in forecasts.columns
]

forecasts = forecasts.reset_index()


# 4. Ühendame tegelikud mõõtmised prognoosidega
combined = actual.merge(
    forecasts,
    on="time",
    how="inner",
    validate="one_to_one"
)

combined = combined.sort_values("time").reset_index(drop=True)

# 5. Salvestame lõpliku andmestiku
combined.to_csv(
    kaust / "weather_dataset_2024_2026.csv",
    index=False
)

print("Lõplik andmestik:", combined.shape)
print("\nPuuduvad väärtused:")
print(combined.isna().sum())

display(combined.head())


Tegelikud mõõtmised: 6624
Lõplik andmestik: (6624, 13)

Puuduvad väärtused:
time                     0
actual_temperature       0
actual_precipitation     0
actual_wind_speed        0
temperature_ECMWF        0
temperature_GFS          0
temperature_ICON_EU      0
precipitation_ECMWF      0
precipitation_GFS        0
precipitation_ICON_EU    0
wind_speed_ECMWF         0
wind_speed_GFS           0
wind_speed_ICON_EU       0
dtype: int64


,time,actual_temperature,actual_precipitation,actual_wind_speed,temperature_ECMWF,temperature_GFS,temperature_ICON_EU,precipitation_ECMWF,precipitation_GFS,precipitation_ICON_EU,wind_speed_ECMWF,wind_speed_GFS,wind_speed_ICON_EU
0,2024-06-01 00:00:00,17.4,0.0,0.8,17.2,15.5,17.4,0.0,1.0,0.0,1.90,2.66,1.04
1,2024-06-01 01:00:00,17.1,0.0,1.2,16.4,15.6,16.5,0.0,0.0,0.0,1.30,2.19,1.13
2,2024-06-01 02:00:00,15.3,0.0,1.3,16.0,15.5,16.0,0.0,0.0,0.0,0.90,1.71,1.08
3,2024-06-01 03:00:00,14.5,0.0,0.7,16.2,15.1,15.6,0.3,0.0,0.0,1.03,2.18,1.00
4,2024-06-01 04:00:00,14.5,0.0,0.9,16.8,15.0,15.3,0.3,0.0,0.0,1.63,2.48,0.61
